In [40]:
# Build per-document commodity vectors directly from dataset topic labels (no text matching).
COMMODITY_TOPIC_CODES = ["M14", "M141", "M142", "M143"]

def extract_dataset_commodity_vector(topics):
    if not isinstance(topics, list):
        return []
    return [code for code in COMMODITY_TOPIC_CODES if code in topics]

def add_dataset_commodity_vectors(df):
    out = df.copy()
    out["commodity_vector"] = out["topics"].apply(extract_dataset_commodity_vector)
    out["commodity_vector_names"] = out["commodity_vector"].apply(
        lambda codes: [topic_label_map.get(code, code) for code in codes]
    )
    out["commodity_count"] = out["commodity_vector"].apply(len)
    return out

rcv1_df = add_dataset_commodity_vectors(rcv1_df)
rcv1_train_df = add_dataset_commodity_vectors(rcv1_train_df)
rcv1_test_df = add_dataset_commodity_vectors(rcv1_test_df)

def _dataset_label_distribution(df):
    exploded = df.explode("commodity_vector").dropna(subset=["commodity_vector"])
    counts = exploded["commodity_vector"].value_counts().rename_axis("commodity_label").reset_index(name="doc_count")
    counts["doc_share_%"] = (counts["doc_count"] / len(df) * 100).round(3)
    counts["description"] = counts["commodity_label"].map(topic_label_map).fillna("")
    return counts

full_commodity_dist = _dataset_label_distribution(rcv1_df)
train_commodity_dist = _dataset_label_distribution(rcv1_train_df)
test_commodity_dist = _dataset_label_distribution(rcv1_test_df)

print("=" * 90)
print("COMMODITY VECTORS FROM DATASET TOPIC LABELS")
print("=" * 90)
print("Full dataset: commodity topic distribution")
display(full_commodity_dist)
print("\nTrain split: commodity topic distribution")
display(train_commodity_dist)
print("\nTest split: commodity topic distribution")
display(test_commodity_dist)

sample_cols = [
    "title",
    "item_id",
    "date_dir",
    "topics",
    "commodity_vector",
    "commodity_vector_names",
    "commodity_count",
]
print("\nSample documents with dataset commodity vectors")
display(rcv1_df[sample_cols].loc[rcv1_df["commodity_count"] > 0].head(10))

print("\nNote: RCV1 topic labels provide broad commodity categories (M14/M141/M142/M143), not specific commodities like wheat/gold per document.")

COMMODITY VECTORS FROM DATASET TOPIC LABELS
Full dataset: commodity topic distribution


,commodity_label,doc_count,doc_share_%,description
0,M14,85100,10.548,COMMODITY MARKETS
1,M141,47708,5.913,SOFT COMMODITIES
2,M143,21957,2.722,ENERGY MARKETS
3,M142,12136,1.504,METALS TRADING



Train split: commodity topic distribution


,commodity_label,doc_count,doc_share_%,description
0,M14,2518,10.877,COMMODITY MARKETS
1,M141,1508,6.514,SOFT COMMODITIES
2,M143,606,2.618,ENERGY MARKETS
3,M142,311,1.343,METALS TRADING



Test split: commodity topic distribution


,commodity_label,doc_count,doc_share_%,description
0,M14,82582,10.570,COMMODITY MARKETS
1,M141,46200,5.913,SOFT COMMODITIES
2,M143,21351,2.733,ENERGY MARKETS
3,M142,11819,1.513,METALS TRADING



Sample documents with dataset commodity vectors


,title,item_id,date_dir,topics,commodity_vector,commodity_vector_names,commodity_count
5,"USA: Hog prices tumble as supplies increase, c...",2291,19960820,"[M14, MCAT]",[M14],[COMMODITY MARKETS],1
6,USA: Blue chips end up as Fed keeps interest r...,2292,19960820,"[M11, M12, M13, M132, M14, MCAT]",[M14],[COMMODITY MARKETS],1
21,UK: Oil prices slip as refiners shop for barga...,2307,19960820,"[M14, M143, MCAT]","[M14, M143]","[COMMODITY MARKETS, ENERGY MARKETS]",2
36,"UK: Silver hits two-month peak, gold edges hig...",2322,19960820,"[M14, M142, MCAT]","[M14, M142]","[COMMODITY MARKETS, METALS TRADING]",2
65,"UK: Silver fixes at two-month high, but gold l...",2355,19960820,[M142],[M142],[METALS TRADING],1
78,"UK: Silver hits two-month peak, gold edges hig...",2374,19960820,[M142],[M142],[METALS TRADING],1
118,HONG KONG: China misses major corn export oppo...,2439,19960820,[M141],[M141],[SOFT COMMODITIES],1
123,THAILAND: Thai rice traders greet expected end...,2446,19960820,"[C13, C31, C311, C312, CCAT, M14, M141, MCAT]","[M14, M141]","[COMMODITY MARKETS, SOFT COMMODITIES]",2
320,CZECH REPUBLIC: CZECH CABINET TO PAN GRAIN PRI...,2682,19960820,"[C13, C31, CCAT, M14, M141, MCAT]","[M14, M141]","[COMMODITY MARKETS, SOFT COMMODITIES]",2
336,USA: U.S. STOCKS MIXED IN VERY SLOW TRADING.,2698,19960820,"[M11, M12, M13, M132, M14, MCAT]",[M14],[COMMODITY MARKETS],1



Note: RCV1 topic labels provide broad commodity categories (M14/M141/M142/M143), not specific commodities like wheat/gold per document.
